# Čovjek-u-petlji: Vrata prije akcije, rangiranje rizika i zapisnik nadzora

README za ovu lekciju uvodi Čovjeka-u-petlji sa kratkim isječkom koji traži od korisnika da `ODOBRI` ili `ODBIJE` nakon što je agent već dao odgovor. Taj obrazac je dobar početak, ali stvarne HITL implementacije često trebaju još tri dodatna elementa:

1. **Vrata prije akcije** koja se pokreću **prije** nego agent izvrši rizični korak, kako bi se troškovi, nepovratnost i latencija držali pod kontrolom.
2. **Rangiranje rizika**, tako da se radnje niskog rizika izvršavaju automatski, radnje srednjeg rizika odobravaju u grupama, a samo radnje visokog rizika blokiraju se dok ih čovjek ne pregleda.
3. **Zapisnik nadzora plus petlja revizije**, tako da se svaka odluka o vratima bilježi kao JSONL, a odbijanje ponovno pokreće agenta s strukturiranim razlogom umjesto da samo ispiše `Revising...`.

Ovaj bilježnik gradi svaki od ovih elemenata na istim primitivima kao `06-system-message-framework.ipynb`. Izvodi se end-to-end u `DEMO_MODE = True` (nije potreban interaktivni unos) ili s pravim `input()` upitima kada je `DEMO_MODE = False`. Napomena: u DEMO_MODE treći cilj za ponovni pokušaj je skriptiran tako da su mehanike petlje vidljive od početka do kraja. Prava revizijsko-temeljena rekategorizacija zahtijeva `DEMO_MODE = False` i operatera.

**Izvan opsega (obrađuje se u drugim lekcijama):** autentikacija i kontrola pristupa (lekcija 06 README prijetnja 2), middleware za poziv alata (lekcija 14 MAF dubinski pregled), obrasci višestrukih agenata za raspravu.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Uzorak 1: Vrata prije akcije

HITL isječak iz README-a najprije poziva agenta, a zatim traži od korisnika da odobri izlaz. To je **tok nakon akcije**. Agent je već izvršio radnju, pa je trošak poziva LLM-a već plaćen, a bilo koji nuspojav (poslani email, upisan red u bazu podataka, objavljeni komentar) već se dogodio.

**Tok prije akcije** umeće vrata prije nego što agent izvrši rizični korak. Agent predlaže akciju, vrata odlučuju hoće li se izvršiti, i nuspojava se dogodi samo ako je odobreno.

| Aspekt | Odobrenje nakon akcije (isječak iz README-a) | Vrata prije akcije (ovaj bilježnik) |
|---|---|---|
| Kada se pokreće odobrenje? | Nakon što je agent proizveo izlaz | Prije izvršenja bilo koje nuspojave |
| Trošak LLM-a kod odbijanja | Već je plaćen | Plaća se samo za prijedlog, ne i za akciju |
| Nepovratne nuspojave | Moguće (akcija se već dogodila) | Spriječeno |
| Jasnoća revizije | Odobrenje je ispisna naredba | Odobrenje je JSON zapis s vremenskom oznakom, akcijom, razlogom |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Uzorak 2: Razvrstavanje prema riziku

Nije svaka radnja potrebna ljudska potvrda. Pregled samo za čitanje preko javnog API-ja ima drugačiji rizik nego slanje e-pošte kupcu. Postupanje s obje isto troši pažnju operatera i usporava agenta.

Jednostavan model s 3 razine:

| Razina | Primjeri | Tijek odobrenja |
|---|---|---|
| `low` (samo čitanje) | Pretraživanje baze znanja, pregled opcija leta, dohvaćanje javne web stranice | Automatsko izvršenje, evidentirano za reviziju |
| `medium` (jeftine promjene) | Keširanje rezultata, povećanje brojača, zakazivanje podsjetnika | Automatsko izvršenje, ali se pregledava u dnevnim serijama |
| `high` (usmjereno prema van ili nepovratno) | Slanje e-pošte, naplata kartice, objava na javnom kanalu | Blokira se dok ne dobije ljudsko odobrenje |

Ovo je jedno razvrstavanje po razinama. Produkcijski sustavi često koriste detaljnije razine (npr. AWS IAM razine dopuštenja, razine pristupa temeljene na ulozi). Verzija s 3 razine ispod je najmanja korisna verzija za agenta koji kombinira radnje samo za čitanje i one koje mijenjaju stanje.

Klasiifikator u nastavku koristi heuristiku ključnih riječi kako bi demo ostao determinističan i jeftin. U produkcijskom sustavu zamijenili biste to naučenim klasifikatorom ili sustavom pravila.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Uzorak 3: Dnevnik revizije i petlja revizije

`print("Response approved.")` nije zapisnik revizije. Za povjerenje, svaka odluka vrata trebala bi biti zabilježena kao strukturirani događaj koji kasnije možete upitavati, reproducirati ili priložiti pregledu incidenta.

Dvije stvari:

1. **Samo dodavanje JSONL.** Jedan red po odluci, s vremenskim žigom, akcijom, razinom, odlukom, razlogom. Lako se pretražuje (grep), lako se kasnije šalje u pravi zapisnik.
2. **Petlja revizije pri odbijanju.** Kada vrata vrate `deny`, agent sam sebe ponovno pita s razlogom odbijanja u kontekstu, tako da sljedeći prijedlog može izbjeći problem.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatni resursi

Nekoliko drugih javnih projekata implementira varijacije ovih HITL obrazaca. Usporedite pristupe kako biste pronašli što odgovara vašem stacku:

- **LangChain** omotači alata s čovjekom u petlji ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): omotači alata koji pauziraju izvođenje za unos čovjeka.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ je restrukturiran): koristi ulogu agenta posebno za predstavljanje čovjeka u višestrukim konverzacijama agenata.
- **Microsoft Agent Framework (MAF)** middleware za pozivanje funkcija ([docs](https://learn.microsoft.com/agent-framework/)): middleware koji se izvršava oko svakog poziva alata/funkcije, pogodan za logiku odobrenja i tokove kontrole.

Svaki projekt drugačije rukuje trima pod-obrascima: LangChain ih obavija kao alate, AutoGen koristi ulogu agenta, a Microsoft Agent Framework koristi middleware za pozivanje funkcija. Pročitajte jednu ili dvije implementacije u cijelosti prije nego što odaberete dizajn za vlastitog agenta.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
